In [79]:
import pandas as pd
import spacy

df = pd.read_csv("DataCollection/PostTokenizing/custom-imdb.csv")
df2 = pd.read_csv("DataCollection/PostTokenizing/45000-cleaned.csv")
df = pd.concat([df, df2[df2["Year"] > 2010]])
df.drop_duplicates(inplace=True)


In [80]:
import DataProcessing.tokenizing as tokenizing


In [81]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [82]:
# === Step 2: Define input and output ===
X = df[["Description", "Keywords"]]
y = df["Title"]

# === Step 3: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [83]:
# === Step 4: ColumnTransformer for multi-input text ===
preprocessor = ColumnTransformer(
    transformers=[
        ("desc_tfidf", TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words='english'), "Description"),
        ("kw_tfidf", TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english'), "Keywords")
    ]
)

In [84]:
# === Step 5: Classifier pipeline ===
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [85]:
# === Step 6: Fit the model ===
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('desc_tfidf',
                                                  TfidfVectorizer(max_features=1000,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  stop_words='english'),
                                                  'Description'),
                                                 ('kw_tfidf',
                                                  TfidfVectorizer(max_features=500,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  stop_words='english'),
                                                  'Keywords')])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [86]:
# === Step 7: Evaluate ===
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.0011534025374855825
                                                                       precision    recall  f1-score   support

                                                              #Horror       0.00      0.00      0.00         1
                                              'Til Madness Do Us Part       0.00      0.00      0.00         1
                                                        10 000 timmar       0.00      0.00      0.00         1
                                                          10,000 Days       0.00      0.00      0.00         1
                              100 Yen: The Japanese Arcade Experience       0.00      0.00      0.00         1
                                                           1001 Grams       0.00      0.00      0.00         1
                                                         112 Weddings       0.00      0.00      0.00         1
                                                12 Rounds 2: Reloaded       0.0

C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

In [89]:
# === Step 8: Predict new sample ===
def predict_title(description):
    input_df = pd.DataFrame({"Description": [description], "Keywords": [" ".join(tokenizing.get_keywords(description))]})
    return pipeline.predict(input_df)[0]

Text = """A teenager accidentally travels back
in time to the 1950s, encountering his young parents and
struggling to return to his own time without disrupting the
future."""
# Example usage
print(predict_title(Text))

print(predict_title("Logan is old, and meets a female clone of his"))

Eden
Hotel Transylvania 2


In [88]:
print(str(df[df["Title"] == "The Bridge"]["Description"].iloc[0]))


Karen Kingsbury's The Bridge is the sweeping tale of Molly Callens (Findlay) and Ryan Kelly (Nash), two young students who share a profound friendship their first semester in college, a time that becomes the defining moment of their lives.
